In [58]:
import os

target_dir = r'data' 
if os.path.exists(target_dir): 
    print("Folder found! Files inside:") 
    print(os.listdir(target_dir)) 
else: 
    print("Folder NOT found. Check the path again.")

Folder found! Files inside:
['apartments_for_rent_classified_10K.csv', 'processed']


In [59]:
import pandas as pd 
file_path = r'data/apartments_for_rent_classified_10K.csv'
try:
    df_apartments = pd.read_csv(file_path, sep=';', encoding='ISO-8859-1')
    print("File loaded successfully. Here's a preview:")
    print(f"数据量：{df_apartments.shape[0]} 行, {df_apartments.shape[1]} 列")
    display(df_apartments.head())
except Exception as e:
    print(f"Error loading file: {e}")

File loaded successfully. Here's a preview:
数据量：10000 行, 22 列


,id,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,5668626895,housing/rent/apartment,"Studio apartment 2nd St NE, Uhland Terrace NE,...","This unit is located at second St NE, Uhland T...",NaN,NaN,0.0,USD,No,Thumbnail,...,$790,Monthly,101,NaN,Washington,DC,38.9057,-76.9861,RentLingo,1577359415
1,5664597177,housing/rent/apartment,Studio apartment 814 Schutte Road,"This unit is located at 814 Schutte Road, Evan...",NaN,NaN,1.0,USD,No,Thumbnail,...,$425,Monthly,106,814 Schutte Rd,Evansville,IN,37.9680,-87.6621,RentLingo,1577017063
2,5668626833,housing/rent/apartment,"Studio apartment N Scott St, 14th St N, Arling...","This unit is located at N Scott St, 14th St N,...",NaN,1.0,0.0,USD,No,Thumbnail,...,"$1,390",Monthly,107,NaN,Arlington,VA,38.8910,-77.0816,RentLingo,1577359410
3,5659918074,housing/rent/apartment,Studio apartment 1717 12th Ave,"This unit is located at 1717 12th Ave, Seattle...",NaN,1.0,0.0,USD,No,Thumbnail,...,$925,Monthly,116,1717 12th Avenue,Seattle,WA,47.6160,-122.3275,RentLingo,1576667743
4,5668626759,housing/rent/apartment,"Studio apartment Washington Blvd, N Cleveland ...","This unit is located at Washington Blvd, N Cle...",NaN,NaN,0.0,USD,No,Thumbnail,...,$880,Monthly,125,NaN,Arlington,VA,38.8738,-77.1055,RentLingo,1577359401


In [60]:
df = df_apartments

import pandas as pd
import re
import numpy as np
# -------------------------
# 0) 你关心的“设施关键词词表”
#    先用你截图里的这些（可以后续继续加/改）
# -------------------------
AMENITY_KEYWORDS = [
    "parking", "dishwasher", "pool", "refrigerator", "patio", "deck",
    "cable", "satellite", "storage", "gym", "internet", "clubhouse",
    "garbage disposal", "washer", "dryer", "fireplace", "playground",
    "ac", "elevator", "tennis", "gated", "wood floors", "hot tub",
    "basketball", "tv", "view", "doorman", "alarm", "golf", "luxury"
]

# 为了更稳：把类似 patio/deck 拆成两个词
# 你也可以自己按需要扩展同义词
KEYWORD_ALIASES = {
    "patio": ["patio", "patio/deck", "deck"],
    "deck": ["deck", "patio/deck", "patio"],
    "washer": ["washer", "laundry"],
    "dryer": ["dryer", "laundry"],
    "ac": ["ac", "a/c", "air conditioning"],
    "internet": ["internet", "wi-fi", "wifi"],
    "cable": ["cable", "satellite", "cable or satellite"]
}

# 把 aliases 展开为最终关键词集合
expanded_keywords = set()
for kw in AMENITY_KEYWORDS:
    if kw in KEYWORD_ALIASES:
        expanded_keywords.update(KEYWORD_ALIASES[kw])
    else:
        expanded_keywords.add(kw)

# 按长度排序，优先匹配长词（如 "garbage disposal"）
expanded_keywords = sorted(expanded_keywords, key=len, reverse=True)

# 构建正则：单词边界匹配（避免 carpool 匹配 pool）
# 注：包含空格的短语也能匹配
pattern = re.compile(r"(?i)\b(" + "|".join(map(re.escape, expanded_keywords)) + r")\b")


# -------------------------
# 1) 字段准备 + 缺失修复
# -------------------------
for col in ["amenities", "body"]:
    if col not in df.columns:
        df[col] = ""

df["amenities_raw"] = df["amenities"]
df["body_raw"] = df["body"]

df["amenities_clean"] = (
    df["amenities"]
    .fillna("")
    .astype(str)
    .str.replace(",", ", ", regex=False)  # 先统一逗号格式，避免 "pool, gym" 和 "pool,gym" 匹配不一致
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["body_clean"] = (
    df["body"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


# -------------------------
# 2) 从文本提取关键词（返回去重后的关键词列表字符串）
# -------------------------
def extract_amenities(text: str) -> str:
    if not text:
        return ""
    found = pattern.findall(text)
    # 统一小写并去重
    found = sorted(set([f.lower() for f in found]))
    return ", ".join(found) if found else ""


df["amenities_from_amenities"] = df["amenities_clean"].apply(extract_amenities)
df["amenities_from_body"] = df["body_clean"].apply(extract_amenities)


# -------------------------
# 3A) Strategy A: only fill from body when amenities is empty
# -------------------------
amenities_empty_mask = df["amenities_from_amenities"].eq("")

df["amenities_filled_A"] = df["amenities_from_amenities"].copy()
df.loc[amenities_empty_mask, "amenities_filled_A"] = df.loc[amenities_empty_mask, "amenities_from_body"]


# -------------------------
# 3B) Strategy B: always merge (union) keywords from amenities + body
# -------------------------
def merge_keywords(a: str, b: str) -> str:
    a_set = set([x.strip() for x in str(a).split(",") if x.strip()])
    b_set = set([x.strip() for x in str(b).split(",") if x.strip()])
    merged = sorted(a_set | b_set)
    return ", ".join(merged)

df["amenities_filled_B"] = [
    merge_keywords(a, b)
    for a, b in zip(df["amenities_from_amenities"], df["amenities_from_body"])
]


# -------------------------
# 4) 生成 yes/no 标记（A / B）
# -------------------------
df["amenities_flag_A"] = np.where(df["amenities_filled_A"].ne(""), "yes", "no")
df["amenities_flag_B"] = np.where(df["amenities_filled_B"].ne(""), "yes", "no")

extra_yes = ((df["amenities_flag_B"] == "yes") & (df["amenities_flag_A"] == "no")).sum()
print("B adds YES over A:", int(extra_yes))
# -------------------------
# 5) 输出结果：统计 + 样例展示（A / B）
# -------------------------
print("=== Strategy A (fill only when amenities empty) ===")
print(df["amenities_flag_A"].value_counts(dropna=False))
print("Ratio:\n", df["amenities_flag_A"].value_counts(normalize=True))

print("\n=== Strategy B (merge amenities + body always) ===")
print(df["amenities_flag_B"].value_counts(dropna=False))
print("Ratio:\n", df["amenities_flag_B"].value_counts(normalize=True))

# B 相比 A 多出来的 YES（A是no，但B变成yes）
extra_yes = ((df["amenities_flag_B"] == "yes") & (df["amenities_flag_A"] == "no")).sum()
print("\nB adds YES over A:", int(extra_yes))

# -------------------------
# 6) 额外分析：B 比 A 多补了哪些关键词
# -------------------------
def set_from_string(s):
    return set([x.strip() for x in str(s).split(",") if x.strip()])

df["extra_in_B"] = [
    ", ".join(sorted(set_from_string(b) - set_from_string(a)))
    for a, b in zip(df["amenities_filled_A"], df["amenities_filled_B"])
]

print("\n=== Examples: extra keywords added by B (top 10) ===")
examples_extra = df[df["extra_in_B"].ne("")][
    ["amenities_raw", "amenities_from_body", "amenities_filled_A", "amenities_filled_B", "extra_in_B"]
].head(10)
print(examples_extra.to_string(index=False))

print("\n=== Examples: A filled from body because amenities was empty (top 10) ===")
examples_filled_A = df[amenities_empty_mask & (df["amenities_flag_A"] == "yes")][
    ["amenities_raw", "body_raw", "amenities_from_body", "amenities_filled_A"]
].head(10)
print(examples_filled_A.to_string(index=False))

print("\n=== Examples: B is yes but A is no (top 10) ===")
examples_B_yes_A_no = df[(df["amenities_flag_B"] == "yes") & (df["amenities_flag_A"] == "no")][
    ["amenities_raw", "body_raw", "amenities_from_amenities", "amenities_from_body", "amenities_filled_A", "amenities_filled_B", "extra_in_B"]
].head(10)
print(examples_B_yes_A_no.to_string(index=False))

print("\n=== Examples: no keywords found by B (top 10) ===")
examples_no_B = df[df["amenities_flag_B"] == "no"][
    ["amenities_raw", "body_raw", "amenities_from_body", "amenities_filled_B"]
].head(10)
print(examples_no_B.to_string(index=False))

# =============================
# Extra: Compare empty definitions
# =============================

# 1️⃣ 字段层面的空（NaN 或 空字符串）
field_empty = df["amenities"].isna() | (df["amenities"].astype(str).str.strip() == "")

# 2️⃣ 关键词层面的空（没有识别到任何关键词）
keyword_empty = df["amenities_from_amenities"].astype(str).str.strip().eq("")

print("===== Empty Definition Comparison =====")
print("Field-empty (NaN or ''):", int(field_empty.sum()))
print("Keyword-empty (no detected keywords):", int(keyword_empty.sum()))

print("\nRows where field NOT empty but keyword empty:",
      int((~field_empty & keyword_empty).sum()))

print("Rows where field empty but keyword NOT empty:",
      int((field_empty & ~keyword_empty).sum()))

examples = df.loc[(~field_empty & keyword_empty),
                  ["amenities", "amenities_from_amenities"]].head(10)

#print("\n===== Example: Field not empty but keyword empty =====")
#print(examples.to_string(index=False))

# =========================
# Amenity Extraction (Synonym Map) — Notebook-ready
# - Does NOT overwrite original columns
# - Extracts standardized amenity labels from BOTH amenities + body
# - Supports Strategy A (fill only when amenities empty) vs Strategy B (always union)
# - Prints summary stats + examples
# =========================

import re
import pandas as pd
import numpy as np

# ---- 0) Synonym map (regex -> canonical label)
synonym_map = {
    # Parking
    r'covered parking|garage|carport|assigned parking|off-street parking|valet': 'Parking',

    # Cooling
    r'central air|air conditioning|ac|a/c|central ac|central a/c': 'AC',

    # Laundry
    r'w/d|washer/dryer|washer and dryer|in-unit laundry|laundry room|washer|dryer|laundry': 'Washer Dryer',

    # Fitness / Pool
    r'fitness center|health club|gym|fitness|exercise room|weight room': 'Gym',
    r'swimming pool|pool|swimming|heated pool|lap pool': 'Pool',

    # Kitchen
    r'dishwasher|dw': 'Dishwasher',
    r'refrigerator|fridge|freezer': 'Refrigerator',
    r'garbage disposal|disposal|sink disposal': 'Garbage Disposal',

    # Outdoor & Living
    r'patio|deck|balcony|terrace|private outdoor space': 'Patio/Deck',
    r'fireplace|gas fireplace|wood-burning fireplace': 'Fireplace',
    r'wood floors|hardwood|hardwood flooring|laminate floors': 'Wood Floors',
    r'view|city view|mountain view|skyline|waterfront': 'View',

    # Community
    r'clubhouse|rec room|recreation center|resident lounge': 'Clubhouse',
    r'playground|play area|tot lot': 'Playground',
    r'elevator|lift': 'Elevator',
    r'doorman|concierge|attended lobby': 'Doorman',
    r'gated|gated community|controlled access|secure entry': 'Gated',

    # Connectivity & Entertainment
    r'cable ready|satellite|cable or satellite|directv|comcast': 'Cable or Satellite',
    r'internet access|high speed internet|wifi|wi-fi|broadband': 'Internet Access',
    r'tv|television': 'TV',

    # Luxury & Sports
    r'hot tub|jacuzzi|spa|whirlpool': 'Hot Tub',
    r'tennis|tennis court': 'Tennis',
    r'basketball|basketball court': 'Basketball',
    r'golf|golf course|putting green': 'Golf',
    r'storage|extra storage|storage unit|locker': 'Storage',
    r'alarm|security system|burglar alarm': 'Alarm',
    r'luxury': 'Luxury'
}

# ---- 1) Make a safe working copy (recommended)
# If you already have df_raw, use df = df_raw.copy(deep=True)
# df = df.copy(deep=True)

# ---- 2) Create clean text columns (do NOT overwrite originals)
for col in ["amenities", "body"]:
    if col not in df.columns:
        df[col] = np.nan

df["amenities_raw"] = df["amenities"]
df["body_raw"] = df["body"]

df["amenities_clean"] = (
    df["amenities"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["body_clean"] = (
    df["body"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# ---- 3) Compile patterns (with boundaries to reduce false matches)
# Note: we add word boundaries around the whole group.
compiled_patterns = []
for pat, label in synonym_map.items():
    # safer boundary: avoid matching inside words (e.g., "carpool" shouldn't match "pool")
    # Still not perfect for every case, but much better than substring matching.
    regex = re.compile(rf"(?i)(?<!\w)({pat})(?!\w)")
    compiled_patterns.append((regex, label))

def extract_amenities_syn(text: str) -> str:
    """Return standardized labels found in text, as a sorted comma-separated string."""
    if not isinstance(text, str) or text.strip() == "":
        return ""
    found = set()
    for rx, label in compiled_patterns:
        if rx.search(text):
            found.add(label)
    return ", ".join(sorted(found)) if found else ""

# ---- 4) Extract canonical amenities from both sources
df["amenities_from_amenities_syn"] = df["amenities_clean"].apply(extract_amenities_syn)
df["amenities_from_body_syn"] = df["body_clean"].apply(extract_amenities_syn)

# ---- 5) Strategy A vs B
# Strategy A: fill from body only when amenities-side extraction is empty
mask_A = df["amenities_from_amenities_syn"].eq("")
df["amenities_filled_A_syn"] = df["amenities_from_amenities_syn"].copy()
df.loc[mask_A, "amenities_filled_A_syn"] = df.loc[mask_A, "amenities_from_body_syn"]

# Strategy B: always union (amenities ∪ body)
def merge_labels(a: str, b: str) -> str:
    a_set = set([x.strip() for x in str(a).split(",") if x.strip()])
    b_set = set([x.strip() for x in str(b).split(",") if x.strip()])
    merged = sorted(a_set | b_set)
    return ", ".join(merged) if merged else ""

df["amenities_filled_B_syn"] = [
    merge_labels(a, b)
    for a, b in zip(df["amenities_from_amenities_syn"], df["amenities_from_body_syn"])
]

# Flags
df["amenities_flag_A_syn"] = np.where(df["amenities_filled_A_syn"].ne(""), "yes", "no")
df["amenities_flag_B_syn"] = np.where(df["amenities_filled_B_syn"].ne(""), "yes", "no")

# ---- 6) Summary stats
extra_yes_syn = ((df["amenities_flag_B_syn"] == "yes") & (df["amenities_flag_A_syn"] == "no")).sum()
diff_rows_syn = (df["amenities_filled_B_syn"] != df["amenities_filled_A_syn"]).sum()

def n_labels(s: str) -> int:
    s = str(s).strip()
    if not s:
        return 0
    return len([x for x in s.split(",") if x.strip()])

df["k_A_syn"] = df["amenities_filled_A_syn"].apply(n_labels)
df["k_B_syn"] = df["amenities_filled_B_syn"].apply(n_labels)

print("===== Synonym Map Extraction Summary =====")
print("Total rows:", len(df))
print("\nStrategy A (syn) value counts:")
print(df["amenities_flag_A_syn"].value_counts(dropna=False))
print("\nStrategy B (syn) value counts:")
print(df["amenities_flag_B_syn"].value_counts(dropna=False))

print("\nB adds YES over A (syn):", int(extra_yes_syn))
print("Rows where B differs from A (syn):", int(diff_rows_syn))

print("\nAverage labels A (syn):", float(df["k_A_syn"].mean()))
print("Average labels B (syn):", float(df["k_B_syn"].mean()))
print("Average increase (B-A) (syn):", float((df["k_B_syn"] - df["k_A_syn"]).mean()))
print("Max increase (B-A) (syn):", int((df["k_B_syn"] - df["k_A_syn"]).max()))

# ---- 7) Examples: B adds extra labels beyond A

def extra_labels(a, b):
    a_set = set([x.strip() for x in str(a).split(",") if x.strip()])
    b_set = set([x.strip() for x in str(b).split(",") if x.strip()])
    diff = sorted(b_set - a_set)
    return ", ".join(diff) if diff else ""

df["extra_in_B_syn"] = [
    extra_labels(a, b)
    for a, b in zip(df["amenities_filled_A_syn"], df["amenities_filled_B_syn"])
]

examples_extra = df[df["extra_in_B_syn"].ne("")][
    ["amenities_raw",
     "amenities_from_amenities_syn",
     "amenities_from_body_syn",
     "amenities_filled_A_syn",
     "amenities_filled_B_syn",
     "extra_in_B_syn"]
].head(10)

print("\n===== Examples where B adds more than A (top 10) =====")
print(examples_extra.to_string(index=False))

# ---- 8) Optional: check 'empty' definitions on raw amenities
field_empty = df["amenities_raw"].isna() | (df["amenities_raw"].astype(str).str.strip() == "")
keyword_empty_amen = df["amenities_from_amenities_syn"].eq("")

print("\n===== Empty Definition Comparison (syn) =====")
print("Field-empty (NaN or '') amenities_raw:", int(field_empty.sum()))
print("Keyword-empty on amenities (syn):", int(keyword_empty_amen.sum()))
print("Field not empty but keyword empty:", int((~field_empty & keyword_empty_amen).sum()))

In [61]:
# =========================
# BASELINE MODEL
# =========================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1️⃣ 选择核心变量
features = ["bedrooms", "bathrooms", "square_feet", "latitude", "longitude"]
target = "price"

# 2️⃣ 丢弃缺失
df_model = df[features + [target]].dropna()

print("Baseline sample size:", len(df_model))

X = df_model[features]
y = df_model[target]

# 3️⃣ Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4️⃣ 建立模型
model_baseline = LinearRegression()
model_baseline.fit(X_train, y_train)

# 5️⃣ 预测
y_pred = model_baseline.predict(X_test)

# 6️⃣ 评估
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n===== Baseline Performance =====")
print("RMSE:", round(rmse, 2))
print("R²:", round(r2, 4))

# 7️⃣ 查看系数
coef_table = pd.DataFrame({
    "Feature": features,
    "Coefficient": model_baseline.coef_
})

print("\n===== Baseline Coefficients =====")
print(coef_table)

Baseline sample size: 9950

===== Baseline Performance =====
RMSE: 753.32
R²: 0.3171

===== Baseline Coefficients =====
       Feature  Coefficient
0     bedrooms  -156.174437
1    bathrooms   291.906240
2  square_feet     0.899267
3     latitude     2.052780
4    longitude   -13.558117


In [62]:
df_loc = df.copy(deep=True)

print("Original missing city:", df_loc["cityname"].isna().sum())

Original missing city: 77


In [63]:
# 只用有 city 的数据计算中心
city_centers = (
    df_loc[df_loc["cityname"].notna()]
    .groupby("cityname")[["latitude", "longitude"]]
    .mean()
)

print("Number of city centers:", len(city_centers))

Number of city centers: 1574


import numpy as np

def find_nearest_city(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return np.nan
    
    # 计算与所有城市中心的欧式距离
    distances = np.sqrt(
        (city_centers["latitude"] - lat)**2 +
        (city_centers["longitude"] - lon)**2
    )
    
    return distances.idxmin()

missing_mask = df_loc["cityname"].isna()

df_loc.loc[missing_mask, "city_imputed"] = df_loc[missing_mask].apply(
    lambda row: find_nearest_city(row["latitude"], row["longitude"]),
    axis=1
)

# 非缺失的保持原值
df_loc.loc[~missing_mask, "city_imputed"] = df_loc.loc[~missing_mask, "cityname"]

print("After imputation missing city:", df_loc["city_imputed"].isna().sum())

In [64]:
print("Before:")
print("Missing city:", df['cityname'].isna().sum())
print("Missing state:", df['state'].isna().sum())

Before:
Missing city: 77
Missing state: 77


# !pip install reverse_geocoder
import reverse_geocoder as rg
import numpy as np
import pandas as pd

# 1) 同时识别 NaN 和 空字符串（更稳）
mask = (
    df["cityname"].isna() | df["state"].isna() |
    (df["cityname"].astype(str).str.strip() == "") |
    (df["state"].astype(str).str.strip() == "")
)

to_fix = df.loc[mask, ["latitude", "longitude"]].copy()

# 2) 强制转成数值（无法转换的会变 NaN）
to_fix["latitude"] = pd.to_numeric(to_fix["latitude"], errors="coerce")
to_fix["longitude"] = pd.to_numeric(to_fix["longitude"], errors="coerce")

# 3) 过滤无效坐标：NaN + 超范围
valid = (
    to_fix["latitude"].between(-90, 90) &
    to_fix["longitude"].between(-180, 180)
)

valid_idx = to_fix.index[valid]
invalid_idx = to_fix.index[~valid]

print("Rows needing imputation:", int(mask.sum()))
print("Valid coords for geocode:", len(valid_idx))
print("Invalid coords skipped:", len(invalid_idx))

# 4) 准备 coords
coords = list(zip(to_fix.loc[valid_idx, "latitude"], to_fix.loc[valid_idx, "longitude"]))

# 5) 调用 reverse_geocoder（mode=1 更稳，避免并行worker报错）
if len(coords) > 0:
    results = rg.search(coords, mode=1)  # ✅ 稳定模式

    df.loc[valid_idx, "cityname"] = [r["name"] for r in results]
    df.loc[valid_idx, "state"] = [r["admin1"] for r in results]

print("Imputation complete using offline geocoder.")

In [65]:
print("After:")
print("Missing city:", df['cityname'].isna().sum())
print("Missing state:", df['state'].isna().sum())

After:
Missing city: 77
Missing state: 77


In [66]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# =========================
# 0) 参数区（你可以改）
# =========================
TARGET = "price"
BASE_FEATURES = ["bedrooms", "bathrooms", "square_feet", "latitude", "longitude"]
CITY_COL = "cityname"

RANDOM_STATE = 42
TEST_SIZE = 0.2

# 距离阈值（单位：度）。0.5度大约 ~50km（粗略）
# 阈值越小越保守（更少填补但更少误分）
MAX_DEG_DIST = 0.5

# 低频城市合并阈值（可选，建议先不合并；如果 dummy 太多再启用）
MIN_CITY_COUNT = None  # 例如设成 30 或 50；不想合并就保持 None


# =========================
# 1) 安全副本，不覆盖原数据
# =========================
df_loc = df.copy(deep=True)

# 基础检查
needed_cols = BASE_FEATURES + [TARGET, CITY_COL]
missing_cols = [c for c in needed_cols if c not in df_loc.columns]
if missing_cols:
    raise ValueError(f"df 缺少列：{missing_cols}")

print("Rows:", len(df_loc))
print("Missing city (original):", int(df_loc[CITY_COL].isna().sum()))


# =========================
# 2) 计算城市中心（centroids）
#    只用 city 不缺失 & 经纬度存在的行
# =========================
centroid_source = df_loc.dropna(subset=[CITY_COL, "latitude", "longitude"]).copy()

city_centers = (
    centroid_source
    .groupby(CITY_COL)[["latitude", "longitude"]]
    .mean()
)

print("Unique cities with centroids:", len(city_centers))


# =========================
# 3) 最近城市匹配（含距离阈值）
# =========================
def nearest_city_with_threshold(lat, lon, centers_df, max_deg_dist=0.5):
    """
    返回 (best_city, best_dist)：
    - 如果最近距离 > 阈值，返回 (np.nan, best_dist)
    """
    if pd.isna(lat) or pd.isna(lon) or centers_df.empty:
        return (np.nan, np.nan)

    d = np.sqrt((centers_df["latitude"] - lat)**2 + (centers_df["longitude"] - lon)**2)
    best_city = d.idxmin()
    best_dist = float(d.loc[best_city])

    if best_dist > max_deg_dist:
        return (np.nan, best_dist)
    return (best_city, best_dist)


# 创建新列，不覆盖原 city
df_loc["city_imputed"] = df_loc[CITY_COL]
df_loc["city_impute_dist"] = np.nan

miss_mask = df_loc[CITY_COL].isna()
if miss_mask.any():
    results = df_loc.loc[miss_mask, ["latitude", "longitude"]].apply(
        lambda r: nearest_city_with_threshold(r["latitude"], r["longitude"], city_centers, MAX_DEG_DIST),
        axis=1
    )
    df_loc.loc[miss_mask, "city_imputed"] = results.apply(lambda x: x[0])
    df_loc.loc[miss_mask, "city_impute_dist"] = results.apply(lambda x: x[1])

print("Missing city (after imputation):", int(df_loc["city_imputed"].isna().sum()))
print("Filled city count:", int(miss_mask.sum() - df_loc.loc[miss_mask, "city_imputed"].isna().sum()))

# 看一下被阈值挡掉的（没填上）
blocked = df_loc.loc[miss_mask & df_loc["city_imputed"].isna(), "city_impute_dist"]
if len(blocked) > 0:
    print("Imputation blocked by threshold:", len(blocked))
    print("Blocked dist (min/median/max):",
          round(float(blocked.min()), 4),
          round(float(blocked.median()), 4),
          round(float(blocked.max()), 4))
else:
    print("No rows blocked by threshold.")


# =========================
# 4) 可选：低频城市合并（防止dummy太多）
# =========================
def maybe_collapse_low_freq_city(s: pd.Series, min_count: int | None):
    if min_count is None:
        return s
    vc = s.value_counts(dropna=True)
    keep = vc[vc >= min_count].index
    return s.where(s.isin(keep), other="Other")

df_loc["city_before_for_model"] = maybe_collapse_low_freq_city(df_loc[CITY_COL], MIN_CITY_COUNT)
df_loc["city_after_for_model"] = maybe_collapse_low_freq_city(df_loc["city_imputed"], MIN_CITY_COUNT)


# =========================
# 5) 建模函数：只用 base features + city dummy
#    关键：严格选择列，避免字符串列混入
# =========================
def fit_eval_linear(df_in: pd.DataFrame, city_col_for_model: str, label: str):
    # 只取需要的列
    df_m = df_in[BASE_FEATURES + [TARGET, city_col_for_model]].copy()

    # 必要列缺失就丢掉（保证模型输入一致）
    df_m = df_m.dropna(subset=BASE_FEATURES + [TARGET, city_col_for_model])

    # one-hot city
    df_m = pd.get_dummies(df_m, columns=[city_col_for_model], drop_first=True)

    X = df_m.drop(columns=[TARGET])
    y = df_m[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2 = float(r2_score(y_test, y_pred))

    print(f"\n===== {label} =====")
    print("Sample size:", len(df_m))
    print("RMSE:", round(rmse, 2))
    print("R²:", round(r2, 4))
    print("Num features:", X.shape[1])

    return {"label": label, "n": len(df_m), "rmse": rmse, "r2": r2, "n_features": X.shape[1]}


# =========================
# 6) 对比：imputation 前 vs 后
# =========================

# BEFORE：只使用原始 city（缺失会被 drop）
res_before = fit_eval_linear(df_loc, "city_before_for_model", "BEFORE (drop missing city)")

# AFTER：使用 city_imputed（缺失更少；仍可能有少量被阈值挡掉的缺失）
res_after = fit_eval_linear(df_loc, "city_after_for_model", "AFTER (imputed city by centroid)")

print("\n===== COMPARISON =====")
print("ΔR² (after - before):", round(res_after["r2"] - res_before["r2"], 4))
print("ΔRMSE (after - before):", round(res_after["rmse"] - res_before["rmse"], 2))
print("ΔSample size:", int(res_after["n"] - res_before["n"]))

Rows: 10000
Missing city (original): 77
Unique cities with centroids: 1574
Missing city (after imputation): 76
Filled city count: 1
Imputation blocked by threshold: 76
Blocked dist (min/median/max): 0.8202 0.8202 0.8202

===== BEFORE (drop missing city) =====
Sample size: 9883
RMSE: 504.35
R²: 0.6563
Num features: 1576

===== AFTER (imputed city by centroid) =====
Sample size: 9884
RMSE: 606.09
R²: 0.5686
Num features: 1576

===== COMPARISON =====
ΔR² (after - before): -0.0878
ΔRMSE (after - before): 101.74
ΔSample size: 1
